In [12]:
!pip install pip==23.3.1
!pip install ratsnlp

DEPRECATION: pytorch-lightning 1.6.1 has a non-standard dependency specifier torch>=1.8.*. pip 24.0 will enforce this behaviour change. A possible replacement is to upgrade to a newer version of pytorch-lightning or contact the author to suggest that they release a version with a conforming dependency specifiers. Discussion can be found at https://github.com/pypa/pip/issues/12063
DEPRECATION: pytorch-lightning 1.6.1 has a non-standard dependency specifier torch>=1.8.*. pip 24.0 will enforce this behaviour change. A possible replacement is to upgrade to a newer version of pytorch-lightning or contact the author to suggest that they release a version with a conforming dependency specifiers. Discussion can be found at https://github.com/pypa/pip/issues/12063


In [13]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [14]:
from ratsnlp.nlpbook.ner import NERDeployArguments
args = NERDeployArguments(
    pretrained_model_name="beomi/kcbert-base",
    downstream_model_dir="/content/drive/MyDrive/nlpbook/checkpoint-ner",
    max_seq_length=64,
)

downstream_model_checkpoint_fpath: /content/drive/MyDrive/nlpbook/checkpoint-ner/epoch=1-val_loss=0.11.ckpt
downstream_model_labelmap_fpath: /content/drive/MyDrive/nlpbook/checkpoint-ner/label_map.txt


In [15]:
from transformers import BertTokenizer
tokenizer = BertTokenizer.from_pretrained(
    args.pretrained_model_name,
    do_lower_case=False,
)

In [16]:
import torch
fine_tuned_model_ckpt = torch.load(
    args.downstream_model_checkpoint_fpath,
    map_location=torch.device("cpu"),
)

In [17]:
from transformers import BertConfig, BertForTokenClassification
pretrained_model_config = BertConfig.from_pretrained(
    args.pretrained_model_name,
    num_labels=fine_tuned_model_ckpt['state_dict']['model.classifier.bias'].shape.numel(),
)
model = BertForTokenClassification(pretrained_model_config)

In [18]:
model.load_state_dict({k.replace("model.", ""): v for k, v in fine_tuned_model_ckpt['state_dict'].items()})
model.eval()

BertForTokenClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30000, 768, padding_idx=0)
      (position_embeddings): Embedding(300, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, el

In [19]:
labels = [label.strip() for label in open(args.downstream_model_labelmap_fpath, "r").readlines()]
id_to_label = {}
for idx, label in enumerate(labels):
    if "PER" in label:
      label = "인명"
    elif "LOC" in label:
      label = "지명"
    elif "ORG" in label:
      label = "기관명"
    elif "DAT" in label:
      label = "날짜"
    elif "TIM" in label:
      label = "시간"
    elif "DUR" in label:
      label = "기간"
    elif "MNY" in label:
      label = "통화"
    elif "PNT" in label:
      label = "비율"
    elif "NOH" in label:
      label = "기타수량표현"
    elif "POH" in label:
      label = "기타"
    else:
      label = label
    id_to_label[idx] = label

In [20]:
print(id_to_label)

{0: '[CLS]', 1: '[SEP]', 2: '[PAD]', 3: '[MASK]', 4: 'O', 5: '인명', 6: '기타수량표현', 7: '기타', 8: '기관명', 9: '날짜', 10: '지명', 11: '통화', 12: '비율', 13: '시간', 14: '기간', 15: '인명', 16: '기타수량표현', 17: '기타', 18: '기관명', 19: '날짜', 20: '지명', 21: '통화', 22: '비율', 23: '시간', 24: '기간'}


In [24]:
def inference_fn(sentence):
  inputs = tokenizer(
      [sentence],
      max_length = args.max_seq_length,
      padding = "max_length",
      truncation = True,
  )
  with torch.no_grad():
    outputs = model(**{k: torch.tensor(v) for k, v in inputs.items()})
    probs = outputs.logits[0].softmax(dim = 1)
    top_probs, preds = torch.topk(probs, dim = 1, k = 1)
    tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
    predicted_tags = [id_to_label[pred.item()]for pred in preds]
    result = []
    for token, predicted_tag, top_prob in zip(tokens, predicted_tags, top_probs):
      if token not in [tokenizer.pad_token, tokenizer.cls_token, tokenizer.sep_token]:
        token_result = {
            "token": token,
            "predicted_tag" : predicted_tag,
            "top_prob" : str(round(top_prob[0].item(), 4))
        }
        result.append(token_result)
  return {
      "sentence" : sentence,
      "result" : result,
  }


In [ ]:
from google.colab import output
from ratsnlp.nlpbook.ner import get_web_service_app
from flask import Flask  # 원본 run 메서드

app = get_web_service_app(inference_fn)

# 🔧 패치된 run을 Flask 원본으로 되돌리기
app.run = Flask.run.__get__(app, app.__class__)

# Colab 프록시 URL
PORT = 5000
proxy_url = output.eval_js(f"google.colab.kernel.proxyPort({PORT})")
print("Proxy URL:", proxy_url)

# 이제 인자 먹습니다
app.run(host="0.0.0.0", port=PORT, debug=False, use_reloader=False)

Proxy URL: https://5000-gpu-t4-s-2b2y9pf44j1fr-c.europe-west4-1.prod.colab.dev
 * Serving Flask app 'ratsnlp.nlpbook.ner.deploy'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://172.28.0.12:5000
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug:127.0.0.1 - - [09/Aug/2025 06:01:34] "GET / HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [09/Aug/2025 06:01:42] "POST /api HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [09/Aug/2025 06:01:56] "POST /api HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [09/Aug/2025 06:02:07] "POST /api HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [09/Aug/2025 06:02:12] "POST /api HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [09/Aug/2025 06:02:17] "POST /api HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [09/Aug/2025 06:02:32] "POST /api HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [09/Aug/2025 06:08:13] "POST /api HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [09/Aug/2025 06:08:33] "POST /api HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.